# DJF Data Extraction

Extracts December–January–February (DJF) daily minimum temperature and
precipitation from HadUK-Grid NetCDF files and computes gridded percentile
thresholds used to define compound cold-dry (CD) and cold-wet (CW) events.

**Data source:** HadUK-Grid v1.3.0 (5 km) — available from the CEDA archive:  
https://catalogue.ceda.ac.uk/uuid/4dc8450d889a491ebb20e724debe2dfb

**Configuration:** Edit the paths in the cell below before running.

In [ ]:
import os
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import seaborn as sns

# ── User configuration ──────────────────────────────────────────────────────
# Path to HadUK-Grid tasmin (minimum temperature) directory
data_dir  = "/path/to/HadUK-Grid/5km/tasmin/day/latest/"
# Path to HadUK-Grid rainfall directory
data_dir2 = "/path/to/HadUK-Grid/5km/rainfall/day/latest/"
# Where to save output arrays
output_dir = "output/arrays"
os.makedirs(output_dir, exist_ok=True)

start_year = 1960
end_year   = 2024   # exclusive

LEAP_YEARS = {yr for yr in range(start_year, end_year)
              if (yr % 4 == 0 and yr % 100 != 0) or (yr % 400 == 0)}


## 1. Extract minimum temperature (tasmin)

In [ ]:
# Output array: [years, days-per-season, lat, lon]
# 91 days = 31 (Dec) + 31 (Jan) + 29 (Feb padded)
minT_array  = np.zeros([end_year - start_year, 91, 290, 180])
dates_array = []

counteryearT = 0
for yr in range(start_year, end_year):
    counterT = 0
    for mon in [1, 2, 12]:
        str_mon = str(mon).zfill(2)
        end_day = 31 if mon in [1, 12] else (29 if yr in LEAP_YEARS else 28)
        fname = os.path.join(
            data_dir,
            f"tasmin_hadukgrid_uk_5km_day_{yr}{str_mon}01-{yr}{str_mon}{end_day:02d}.nc"
        )
        ds = Dataset(fname, mode="r")
        lat          = ds.variables["latitude"][:]
        lon          = ds.variables["longitude"][:]
        time         = ds.variables["time"][:]
        min_temp_raw = ds.variables["tasmin"][:]
        ds.close()

        for dayi in range(min_temp_raw.shape[0]):
            minT_array[counteryearT, counterT, :, :] = min_temp_raw[dayi, :, :]
            if mon == 2 and counterT == 28 and yr not in LEAP_YEARS:
                minT_array[counteryearT, dayi + 1, :, :] = np.nan
            dates_array.append(time.data[dayi])
            counterT += 1
    counteryearT += 1

print("Temperature extraction complete. Shape:", minT_array.shape)


## 2. Extract precipitation (rainfall)

In [ ]:
prec_array = np.zeros([end_year - start_year, 91, 290, 180])

counteryearT = 0
for yr in range(start_year, end_year):
    counterT = 0
    for mon in [1, 2, 12]:
        str_mon = str(mon).zfill(2)
        end_day = 31 if mon in [1, 12] else (29 if yr in LEAP_YEARS else 28)
        fname = os.path.join(
            data_dir2,
            f"rainfall_hadukgrid_uk_5km_day_{yr}{str_mon}01-{yr}{str_mon}{end_day:02d}.nc"
        )
        ds = Dataset(fname, mode="r")
        rainfall = ds.variables["rainfall"][:]
        time     = ds.variables["time"][:]
        ds.close()

        for dayi in range(rainfall.shape[0]):
            prec_array[counteryearT, counterT, :, :] = rainfall[dayi, :, :]
            if mon == 2 and counterT == 28 and yr not in LEAP_YEARS:
                prec_array[counteryearT, dayi + 1, :, :] = np.nan
            counterT += 1
    counteryearT += 1

print("Precipitation extraction complete. Shape:", prec_array.shape)


## 3. Compute percentile thresholds

Percentiles are calculated over the most recent 30-year climatological period
(index 34 onward from 1960 = years 1994–2023).

- **15th percentile of minimum temperature** → cold threshold  
- **15th / 85th percentile of wet-day precipitation** (days ≥ 0.2 mm) → dry / wet thresholds

In [ ]:
WET_DAY_THRESHOLD = 0.2  # mm

perc_minT    = np.zeros(lat.shape)
perc_prec_15 = np.zeros(lat.shape)
perc_prec_85 = np.zeros(lat.shape)

for lat_i in range(lat.shape[0]):
    for lon_i in range(lat.shape[1]):
        flat_T = minT_array[34:, :, lat_i, lon_i].flatten()
        flat_P = prec_array[34:, :, lat_i, lon_i].flatten()
        wet    = flat_P[flat_P >= WET_DAY_THRESHOLD]

        perc_minT[lat_i, lon_i] = np.nanpercentile(flat_T, 15)

        if wet.size > 0:
            perc_prec_15[lat_i, lon_i] = np.nanpercentile(wet, 15)
            perc_prec_85[lat_i, lon_i] = np.nanpercentile(wet, 85)
        else:
            perc_prec_15[lat_i, lon_i] = np.nan
            perc_prec_85[lat_i, lon_i] = np.nan

# Uncomment to save:
# np.save(os.path.join(output_dir, "minT_percentiles_30yr.npy"),             perc_minT)
# np.save(os.path.join(output_dir, "perc15_percentiles_30yr_filtered.npy"),  perc_prec_15)
# np.save(os.path.join(output_dir, "perc85_percentiles_30yr_filtered.npy"),  perc_prec_85)
print("Percentile arrays computed.")


## 4. Inspect percentile distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in zip(
    axes,
    [perc_prec_15, perc_prec_85],
    ["15th Percentile of Wet-Day Precipitation",
     "85th Percentile of Wet-Day Precipitation"],
):
    flat = data.flatten().copy()
    flat[flat > 50] = np.nan
    sns.histplot(flat[~np.isnan(flat)], bins=50, kde=True, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Daily Precipitation (mm)")
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()
